### Cell 1: setup, configuration, and constants

In [17]:
# ============================================
# Mastercard IGS Challenge - Fresh Pipeline
# Cell 1: setup, configuration, and constants
# ============================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------
# Project paths
# -----------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for path in [DATA_DIR, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Time windows
# -----------------------------
ALL_YEARS = list(range(2017, 2025))
PRE_COVID_YEARS = [2017, 2018, 2019]
RECOVERY_YEARS = [2022, 2023, 2024]

# -----------------------------
# Core identifiers / columns
# -----------------------------
ID_COLS = [
    "tract_geoid",      # standardized 11-digit tract GEOID
    "state_fips",
    "county_fips",
    "state_name",
    "county_name",
    "year"
]

CORE_IGS_COLS = [
    "igs_total",
    "igs_economy"
]

ACS_SUPPORT_COLS = [
    "poverty_rate",
    "unemp_rate",
    "median_household_income",
    "lfpr_16p"
]

OPTIONAL_CONTEXT_COLS = [
    "total_population"
]

CORE_PANEL_COLS = ID_COLS + CORE_IGS_COLS + ACS_SUPPORT_COLS + OPTIONAL_CONTEXT_COLS

# -----------------------------
# Tract filtering rules
# -----------------------------
LOW_IGS_TOTAL_MAX = 45.0
MIN_VULNERABILITY_FLAGS = 3

# Percentile cutoffs for recovery-period tract averages
VULNERABILITY_RULES = {
    "igs_economy": {
        "direction": "low",
        "percentile": 0.40,
        "flag_name": "flag_low_igs_economy"
    },
    "poverty_rate": {
        "direction": "high",
        "percentile": 0.60,
        "flag_name": "flag_high_poverty"
    },
    "unemp_rate": {
        "direction": "high",
        "percentile": 0.60,
        "flag_name": "flag_high_unemployment"
    },
    "median_household_income": {
        "direction": "low",
        "percentile": 0.40,
        "flag_name": "flag_low_income"
    },
    "lfpr_16p": {
        "direction": "low",
        "percentile": 0.40,
        "flag_name": "flag_low_lfpr"
    }
}

# -----------------------------
# Cluster usability defaults
# -----------------------------
MIN_CLUSTER_TRACTS = 3
MIN_CLUSTER_POP = 5000

# -----------------------------
# Column naming conventions
# -----------------------------
PRE_PREFIX = "pre"
REC_PREFIX = "recovery"
CHG_PREFIX = "chg"

def period_col(period_prefix: str, base_col: str) -> str:
    return f"{period_prefix}_{base_col}"

def change_col(base_col: str) -> str:
    return f"{CHG_PREFIX}_{base_col}"

# -----------------------------
# Helper: standardize GEOID
# -----------------------------
def standardize_tract_geoid(value) -> str:
    """
    Return an 11-digit census tract GEOID as a zero-padded string.
    """
    if pd.isna(value):
        return np.nan
    value = str(value).strip().replace(".0", "")
    digits = "".join(ch for ch in value if ch.isdigit())
    return digits.zfill(11) if digits else np.nan

# -----------------------------
# Helper: safe numeric conversion
# -----------------------------
def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

print("Fresh pipeline configuration loaded.")
print(f"Years: {ALL_YEARS}")
print(f"Pre-COVID window: {PRE_COVID_YEARS}")
print(f"Recovery window: {RECOVERY_YEARS}")
print(f"Low-IGS threshold: recovery_igs_total < {LOW_IGS_TOTAL_MAX}")
print(f"Economic vulnerability rule: at least {MIN_VULNERABILITY_FLAGS} of 5 conditions")

Fresh pipeline configuration loaded.
Years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Pre-COVID window: [2017, 2018, 2019]
Recovery window: [2022, 2023, 2024]
Low-IGS threshold: recovery_igs_total < 45.0
Economic vulnerability rule: at least 3 of 5 conditions


### Cell 2: load raw files and build tract-year panel

In [19]:
# ============================================
# Cell 2: load raw files and build tract-year panel
# ============================================

import geopandas as gpd
from pathlib import Path

# -----------------------------
# File discovery
# Edit these if your filenames differ
# -----------------------------
IGS_CANDIDATES = [
    RAW_DIR / "igs_panel.csv",
    RAW_DIR / "igs.csv",
    RAW_DIR / "Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv",
    Path(r"/mnt/c/Users/jabba/Desktop/Code/machine_learning/AUC_mastercard_challenge/src/Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv"),
]

ACS_CANDIDATES = [
    RAW_DIR / "acs_econ_panel.csv",
    RAW_DIR / "acs_panel.csv",
    RAW_DIR / "acs.csv",
]

TRACT_GEO_CANDIDATES = [
    RAW_DIR / "tract_geometry.geojson",
    RAW_DIR / "tracts.geojson",
    RAW_DIR / "tracts.gpkg",
    RAW_DIR / "tract_shapefile.shp",
]

def pick_existing_file(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None

igs_path = pick_existing_file(IGS_CANDIDATES)
acs_path = pick_existing_file(ACS_CANDIDATES)
tract_geo_path = pick_existing_file(TRACT_GEO_CANDIDATES)

if igs_path is None:
    raise FileNotFoundError(
        f"No IGS file found. Checked: {[str(p) for p in IGS_CANDIDATES]}"
    )

if acs_path is None:
    raise FileNotFoundError(
        f"No ACS file found. Checked: {[str(p) for p in ACS_CANDIDATES]}"
    )

if tract_geo_path is None:
    raise FileNotFoundError(
        f"No tract geometry file found. Checked: {[str(p) for p in TRACT_GEO_CANDIDATES]}"
    )

print("Using files:")
print("IGS:", igs_path)
print("ACS:", acs_path)
print("TRACT GEO:", tract_geo_path)

# -----------------------------
# Load raw files
# -----------------------------
igs_raw = pd.read_csv(igs_path)
acs_raw = pd.read_csv(acs_path)
tract_geo = gpd.read_file(tract_geo_path)

# -----------------------------
# Column alias maps
# Add more aliases here if needed
# -----------------------------
IGS_ALIASES = {
    "tract_geoid": [
        "tract_geoid", "geoid", "GEOID", "tract_id", "tract_fips", "census_tract"
    ],
    "year": [
        "year", "Year", "estimate_year"
    ],
    "state_name": [
        "state_name", "State", "state"
    ],
    "county_name": [
        "county_name", "County", "county"
    ],
    "igs_total": [
        "igs_total", "IGS", "inclusive_growth_score", "overall_score", "igs_overall"
    ],
    "igs_economy": [
        "igs_economy", "economy", "economy_score", "igs_economy_score"
    ],
    "total_population": [
        "total_population", "population", "pop", "B01003_001E"
    ],
}

ACS_ALIASES = {
    "tract_geoid": [
        "tract_geoid", "geoid", "GEOID", "tract_id", "tract_fips", "census_tract"
    ],
    "year": [
        "year", "Year", "estimate_year"
    ],
    "state_name": [
        "state_name", "State", "state"
    ],
    "county_name": [
        "county_name", "County", "county"
    ],
    "poverty_rate": [
        "poverty_rate", "poverty", "poverty_pct", "below_poverty_rate"
    ],
    "unemp_rate": [
        "unemp_rate", "unemployment_rate", "unemployment", "unemp_pct"
    ],
    "median_household_income": [
        "median_household_income", "median_income", "mhi", "hh_income_median"
    ],
    "lfpr_16p": [
        "lfpr_16p", "labor_force_participation_rate", "lfpr", "labor_force_rate"
    ],
    "total_population": [
        "total_population", "population", "pop", "B01003_001E"
    ],
}

TRACT_GEO_ALIASES = {
    "tract_geoid": [
        "tract_geoid", "geoid", "GEOID", "GEOIDFQ", "tract_id", "tract_fips"
    ],
    "state_name": [
        "state_name", "STATE_NAME", "state", "State"
    ],
    "county_name": [
        "county_name", "COUNTY_NAME", "county", "County"
    ],
}

def rename_using_aliases(df, alias_map):
    rename_map = {}
    existing = set(df.columns)
    for canonical, aliases in alias_map.items():
        for alias in aliases:
            if alias in existing:
                rename_map[alias] = canonical
                break
    return df.rename(columns=rename_map)

igs_raw = rename_using_aliases(igs_raw, IGS_ALIASES)
acs_raw = rename_using_aliases(acs_raw, ACS_ALIASES)
tract_geo = rename_using_aliases(tract_geo, TRACT_GEO_ALIASES)

# -----------------------------
# Standardize IDs and types
# -----------------------------
for df in [igs_raw, acs_raw, tract_geo]:
    if "tract_geoid" not in df.columns:
        raise KeyError("A required tract GEOID column was not found after alias matching.")
    df["tract_geoid"] = df["tract_geoid"].apply(standardize_tract_geoid)

if "year" not in igs_raw.columns:
    raise KeyError("IGS file must contain a year column.")
if "year" not in acs_raw.columns:
    raise KeyError("ACS file must contain a year column.")

igs_raw["year"] = to_numeric(igs_raw["year"]).astype("Int64")
acs_raw["year"] = to_numeric(acs_raw["year"]).astype("Int64")

# Keep only analysis years
igs_raw = igs_raw[igs_raw["year"].isin(ALL_YEARS)].copy()
acs_raw = acs_raw[acs_raw["year"].isin(ALL_YEARS)].copy()

# Numeric conversions
for col in ["igs_total", "igs_economy", "total_population"]:
    if col in igs_raw.columns:
        igs_raw[col] = to_numeric(igs_raw[col])

for col in ["poverty_rate", "unemp_rate", "median_household_income", "lfpr_16p", "total_population"]:
    if col in acs_raw.columns:
        acs_raw[col] = to_numeric(acs_raw[col])

# -----------------------------
# Add FIPS pieces from tract GEOID
# -----------------------------
def add_fips_columns(df):
    df = df.copy()
    df["state_fips"] = df["tract_geoid"].str[:2]
    df["county_fips"] = df["tract_geoid"].str[:5]
    return df

igs_raw = add_fips_columns(igs_raw)
acs_raw = add_fips_columns(acs_raw)
tract_geo = add_fips_columns(tract_geo)

# -----------------------------
# Deduplicate tract-year rows
# -----------------------------
igs_raw = (
    igs_raw
    .sort_values(["tract_geoid", "year"])
    .drop_duplicates(subset=["tract_geoid", "year"], keep="last")
    .copy()
)

acs_raw = (
    acs_raw
    .sort_values(["tract_geoid", "year"])
    .drop_duplicates(subset=["tract_geoid", "year"], keep="last")
    .copy()
)

# -----------------------------
# Build tract lookup from geometry
# Keep geometry separate for mapping / adjacency later
# -----------------------------
tract_lookup_cols = ["tract_geoid", "state_fips", "county_fips"]
for col in ["state_name", "county_name"]:
    if col in tract_geo.columns:
        tract_lookup_cols.append(col)

tract_lookup = (
    tract_geo[tract_lookup_cols]
    .dropna(subset=["tract_geoid"])
    .drop_duplicates(subset=["tract_geoid"])
    .copy()
)

# -----------------------------
# Select core columns before merge
# -----------------------------
igs_keep = [c for c in [
    "tract_geoid", "year", "state_fips", "county_fips", "state_name", "county_name",
    "igs_total", "igs_economy", "total_population"
] if c in igs_raw.columns]

acs_keep = [c for c in [
    "tract_geoid", "year", "state_fips", "county_fips", "state_name", "county_name",
    "poverty_rate", "unemp_rate", "median_household_income", "lfpr_16p", "total_population"
] if c in acs_raw.columns]

igs_panel = igs_raw[igs_keep].copy()
acs_panel = acs_raw[acs_keep].copy()

# Avoid duplicate population columns after merge
if "total_population" in acs_panel.columns and "total_population" in igs_panel.columns:
    acs_panel = acs_panel.drop(columns=["total_population"])

# -----------------------------
# Merge into master tract-year panel
# Foundation is IGS, then add ACS support columns
# -----------------------------
tract_year_panel = igs_panel.merge(
    acs_panel,
    on=["tract_geoid", "year"],
    how="left",
    suffixes=("", "_acs")
)

# Fill missing names/FIPS from ACS if needed
for col in ["state_fips", "county_fips", "state_name", "county_name"]:
    acs_col = f"{col}_acs"
    if acs_col in tract_year_panel.columns:
        if col in tract_year_panel.columns:
            tract_year_panel[col] = tract_year_panel[col].fillna(tract_year_panel[acs_col])
        else:
            tract_year_panel[col] = tract_year_panel[acs_col]
        tract_year_panel = tract_year_panel.drop(columns=[acs_col])

# Fill missing names from geometry lookup if still needed
tract_year_panel = tract_year_panel.merge(
    tract_lookup,
    on="tract_geoid",
    how="left",
    suffixes=("", "_geo")
)

for col in ["state_fips", "county_fips", "state_name", "county_name"]:
    geo_col = f"{col}_geo"
    if geo_col in tract_year_panel.columns:
        if col in tract_year_panel.columns:
            tract_year_panel[col] = tract_year_panel[col].fillna(tract_year_panel[geo_col])
        else:
            tract_year_panel[col] = tract_year_panel[geo_col]
        tract_year_panel = tract_year_panel.drop(columns=[geo_col])

# Reorder columns
preferred_order = [
    "tract_geoid", "state_fips", "county_fips", "state_name", "county_name", "year",
    "igs_total", "igs_economy",
    "poverty_rate", "unemp_rate", "median_household_income", "lfpr_16p",
    "total_population"
]
existing_order = [c for c in preferred_order if c in tract_year_panel.columns]
remaining_cols = [c for c in tract_year_panel.columns if c not in existing_order]
tract_year_panel = tract_year_panel[existing_order + remaining_cols].copy()

# Final row-level dedupe
tract_year_panel = (
    tract_year_panel
    .drop_duplicates(subset=["tract_geoid", "year"])
    .reset_index(drop=True)
)

# -----------------------------
# Quick QC
# -----------------------------
print("\nShapes")
print("igs_raw:", igs_raw.shape)
print("acs_raw:", acs_raw.shape)
print("tract_geo:", tract_geo.shape)
print("tract_year_panel:", tract_year_panel.shape)

print("\nPanel years:", sorted(tract_year_panel["year"].dropna().unique().tolist()))
print("Unique tracts in panel:", tract_year_panel["tract_geoid"].nunique())

qc_cols = ["igs_total", "igs_economy", "poverty_rate", "unemp_rate", "median_household_income", "lfpr_16p"]
missing_summary = tract_year_panel[qc_cols].isna().mean().sort_values()
print("\nMissing share by core metric:")
display((missing_summary * 100).round(2).rename("pct_missing").to_frame())

display(tract_year_panel.head())

FileNotFoundError: No IGS file found. Checked: ['C:\\Users\\jabba\\Desktop\\Code\\machine_learning\\AUC_mastercard_challenge\\src\\data\\raw\\igs_panel.csv', 'C:\\Users\\jabba\\Desktop\\Code\\machine_learning\\AUC_mastercard_challenge\\src\\data\\raw\\igs.csv', 'C:\\Users\\jabba\\Desktop\\Code\\machine_learning\\AUC_mastercard_challenge\\src\\data\\raw\\Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv', '\\mnt\\c\\Users\\jabba\\Desktop\\Code\\machine_learning\\AUC_mastercard_challenge\\src\\Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv']